In [ ]:
import sys, os
import glob
import json
import random

import torch
torch.set_default_dtype(torch.float64)
# print(torch.get_num_threads())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import openmm.app as app

sys.path.insert(0, '..')
from cmm.forcefield import CMMForceField
from cmm.units import HARTREE2KCAL
from cmm.develop.data import EdaData, EspData, DipoleData, PolarizabilityData
from cmm.develop.optimize import Trainer, Optimizer
from cmm.develop.metrics import plot_correlation, plot_eda_scan
from cmm.develop.extract_params import extract_bond_angle_eq, extract_multipoles

## Data

In [ ]:
def report_dipos(trainer, data):
    with torch.no_grad():
        res, ref, _, _ = trainer.evaluate([data])
    dipo_cmm = np.linalg.norm(res[0].detach().numpy(), axis=1)
    dipo_qm = np.linalg.norm(ref[0].detach().numpy(), axis=1)
    fig, ax = plt.subplots(1, 1, figsize=(4, 4), constrained_layout=True)
    plot_correlation(dipo_qm, dipo_cmm, 'QM Dipole Moment (a.u.)', 'CMM Dipole Moment (a.u.)', ax=ax)

def report_esp(trainer, data):
    with torch.no_grad():
        res, ref, _, _ = trainer.evaluate([data])
    esp_cmm = res[0].detach().numpy()
    esp_qm = ref[0].detach().numpy()
    mae = np.mean(np.abs(esp_cmm - esp_qm)) * HARTREE2KCAL
    print('MAE (kcal/mol)', mae)


def report_eda(trainer, data, **kwargs):
    with torch.no_grad():
        res, ref, _, _ = trainer.evaluate([data])
    
    if 'k_index' in data.eda_df.columns:
        xdata = data.eda_df['k_index'].values
        xlabel = 'k_index'
    else:
        xdata = data.eda_df['dist'].values
        xlabel = 'dist'
    
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    plot_eda_scan(
        xdata,
        ref[0],
        res[0],
        ax=axes[0],
        **kwargs
    )

    keys = ['perm_elec', 'pauli', 'disp', 'pol', 'ct', 'total']
    with torch.no_grad():
        error = {key: res[0][key] - ref[0][key] for key in keys}
    plot_eda_scan(
        xdata,
        error,
        xlabel=xlabel,
        ylabel='Energy Error (kcal/mol)',
        ax=axes[1],
        **kwargs
    )
    return fig



def report_eda_cluster(trainer, data, **kwargs):
    with torch.no_grad():
        res, ref, _, _ = trainer.evaluate([data])
    
    keys = ['total', 'perm_elec', 'pol', 'ct', 'pauli', 'disp']
    fig, axes = plt.subplots(2, 3, figsize=(9, 6), constrained_layout=True)
    axes = axes.flatten()
    for i in range(len(keys)):
        ax = axes[i]
        key = keys[i]
        plot_correlation(ref[0][key], res[0][key], xlabel='QM (kcal/mol)', ylabel='CMM (kcal/mol)', ax=ax)
        ax.set_title(key.upper())
        

In [ ]:
dimer_datas = [
    EdaData.from_files('dimers/water_water/struct.pdb', glob.glob('dimers/water_water/water_dimer_cluster/*/eda.out')),
    EdaData.from_files('dimers/water_water/struct.pdb', glob.glob('dimers/water_water/2*/*/eda.out')),
]

trimer_outs = [f for f in glob.glob('water_clusters/water_trimer_cluster/*/eda.out') if '1408' not in f]
trimer_data = EdaData.from_files('water_clusters/water_trimer_cluster/struct.pdb', trimer_outs)

tetramer_outs = [f for f in glob.glob('water_clusters/water_tetramer_cluster/*/eda.out') if '1574' not in f]
tetramer_data = EdaData.from_files("water_clusters/water_tetramer_cluster/struct.pdb", tetramer_outs)

pentamer_outs = [f for f in glob.glob('water_clusters/water_pentamer_cluster/*/eda.out') if '1310' not in f and '1005' not in f and '950' not in f and '1072' not in f and '1875' not in f]
pentamer_data = EdaData.from_files("water_clusters/water_pentamer_cluster/struct.pdb", pentamer_outs)
pentamer_data.energies['total'].min()


In [ ]:
def slice_data(data: EdaData, num: int):
    chunks = []
    chunk_size = int(data.num / num)
    for i in range(num):
        chunks.append(chunk_size)
    for i in range(data.num % num):
        chunks[i] += 1
    chunks.insert(0, 0)
    chunks = np.cumsum(chunks)
    return [data[chunks[i]: chunks[i+1]] for i in range(num)]


dimer_datas_batched = dimer_datas[:2]
dimer_datas_batched += slice_data(dimer_datas[-1], 50)
dimer_datas_batched += slice_data(dimer_datas[-2], 50)
random.shuffle(dimer_datas_batched)

## EDA - Optimize Water

In [ ]:
ff = CMMForceField('param_water.json')

optimizer = Optimizer(
    ff,
    freeze_water=False,
    opt_params={
        "atomic_params": [
            'Z', 'b_elec', 
            'b_pauli', 'q_pauli', 'Kdipo_pauli', 'Kquad_pauli', 
            'q_xpol', 'b_xpol', 'Kdipo_xpol', 'Kquad_xpol',
            'C6_disp', 'b_disp'
        ], 
    },
    optim='adam',
    lr=0.05,
    # freeze_types={"atomic_params": ['hw']}
    # l2=1.0,
    # l2_params={'atomic_params': ['Z', 'b_elec']}
)

trainer = Trainer(
    ff, 
    optimizer, 
    target_weights={PolarizabilityData: 1e6, EdaData:1},
    eda_weights={'perm_elec': 10.0, 'total': 0.0, 'pauli': 10.0, 'disp': 100.0, 'pol': 10.0, 'ct': 0.0}
)


report_eda_cluster(trainer, dimer_datas[-2])
report_eda_cluster(trainer, dimer_datas[-1])
report_eda_cluster(trainer, trimer_data)
report_eda_cluster(trainer, tetramer_data)
report_eda_cluster(trainer, pentamer_data)

indices = [i for i, t in enumerate(ff.params['atomic_params']['type']) if t in ['ow', 'hw']]
for key in optimizer.named_params:
    if key.startswith('atomic_params'):
        print(key, ":")
        print(optimizer.named_params[key][indices])

In [ ]:
train_datas = []
for i in range(len(dimer_datas_batched)):
    train_datas.append(dimer_datas_batched[i])

trainer.train_with_batch(train_datas, 10)

In [ ]:

report_eda_cluster(trainer, dimer_datas[-2])
report_eda_cluster(trainer, dimer_datas[-1])

indices = [i for i, t in enumerate(ff.params['atomic_params']['type']) if t in ['ow', 'hw']]
types = [t for i, t in enumerate(ff.params['atomic_params']['type']) if t in ['ow', 'hw']]
print(types)
for key in optimizer.named_params:
    if key.startswith('atomic_params'):
        print(key, ":")
        print(optimizer.named_params[key][indices])


In [ ]:
ff.save('param-opt.json')